# Evaluación, transferencia y límites : fundamentos

**Nivel:** 1  
**Requisitos previos:** 01–04 en orden  
**Tiempo de lectura:** 15 minutos

## Objetivos

El objetivo es explicar el problema y sus datos, inspeccionar y, en niveles superiores, ejecutar el método publicado en su tarea emparejada. Una afirmación útil sobre un controlador requiere denominador válido, resultados independientes, baseline justo, evaluación disjunta y una distribución donde el baseline pueda fallar. Muchos episodios de pocas plantillas fijas no crean muchas situaciones independientes.

## Preparación

Las versiones siguientes están fijadas en `notebooks/requirements.txt` y también constan en los metadatos de este notebook.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Datos

El notebook lee artefactos comprometidos del producto. Propietario: Neuraxis. Repositorio: https://github.com/fsantibanezleal/CAOS_Neuraxis. Licencia: Apache-2.0. El JSON es un artefacto experimental propio y congelado; este notebook no redistribuye datos de benchmarks externos.

Este notebook lee `data/derived/index.json` y las trazas listadas. La ejecución usa un espacio aislado de caso Reflexmesh. Para métodos aprendidos, descarga los activos públicos exactos con `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000` y define `NEURAXIS_CHECKPOINTS` hacia esa carpeta. El manifiesto de publicación verifica los hashes. M09/M10 se leen desde trazas congeladas, salvo que `NEURAXIS_NOTEBOOK_MODEL` indique explícitamente un modelo Ollama local configurado.

Artefactos de esta lección:

- `data/derived/index.json`
- `data/derived/benchmark-adequacy.json`
- `data/derived/semantic-transfer/index.json`
- `data/derived/semantic-transfer/report.json`
- `data/derived/episodes/M01-C01-nominal-40000.json`
- `data/derived/episodes/M04-C01-nominal-40000.json`
- `data/derived/episodes/M09-C01-nominal-40000.json`
- `data/derived/episodes/M12-C01-nominal-40000.json`

## Método

Verifica los digest de origen, compara por separado los carriles central y de transferencia semántica, mide los resultados por método y expone el informe de transferencia emparejado. Las comprobaciones independientes del paquete son autoritativas; coincidir con el profesor, obtener respuesta del planificador o llevar un nombre de método no lo es.

Para una tasa, \(\hat p=\sum_i y_i/n\); el denominador incluye episodios fallidos, inválidos y sin resolver según el contrato del informe. El intervalo de Wilson del índice central es descriptivo bajo independencia entre episodios; las plantillas agrupadas y los resultados idénticos por semilla reducen la información efectiva.

### Comprueba las versiones fijadas del entorno.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verifica la auditoría y los bytes de cada traza antes de interpretar resultados.

In [2]:
# Lee la evidencia comprometida y usa la API publicada; nunca define otra política.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M01', 'M04', 'M09', 'M12']
sample_case = 'C01'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 4 | case: C01 | seed: 40000
Methods: M01, M04, M09, M12


### Inspecciona las decisiones y resultados originales

In [3]:
# Resume la primera acción registrada, los candidatos autorizados y el resultado del oráculo independiente.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M01 | action= a-move | reason= explicit_state_guard | allowed= ['a-ack_event', 'a-move', 'a-refresh', 'a-inspect', 'a-index', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M04 | action= a-move | reason= calibrated_demonstration_preference | allowed= ['a-ack_event', 'a-move', 'a-refresh', 'a-inspect', 'a-index', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M09 | action= a-move | reason= The goal is to route the document to its destination; 'move' is the standard operation for transferring files between locations. | allowed= ['a-ack_event', 'a-move', 'a-refresh', 'a-inspect', 'a-index', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M12 | action= a-move | reason= persistent_outcome_aware_metacontrol | allowed= ['a-ack_event', 'a-move', 'a-refresh', 'a-inspect', 'a-index', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0


## Resultados

La matriz central está saturada: siete métodos resuelven los 1.200 episodios y solo 71/1.440 celdas método/caso/variante cambian por semilla. En transferencia estructural, M04 resuelve 24/24 casos y M12 21/24; el carril semántico opaco mayor también favorece M09 frente a M12. No hay respaldo para afirmar superioridad aprendida.

Este paso de autoría no compromete salidas generadas; `scripts/run_notebooks.py` las ejecuta y guarda desde un kernel nuevo.

## Resumen

El resultado más sólido es un diagnóstico honesto: la matriz central congelada no responde la pregunta de superioridad propuesta. La transferencia es acotada, separada y en comparaciones clave actualmente favorece métodos más simples o alternativos.

## Referencias

- [TOOLSANDBOX](https://arxiv.org/abs/2408.04682)
- [SWE](https://arxiv.org/abs/2310.06770)
- [TAU2](https://arxiv.org/abs/2506.07982)